# 08-10 · M4 준비: 전 기간 총차입금

| 항목 | 내용 |
|---|---|
| 하는 일 | 2018~2024Q3 모든 보고서에서 유동 구간·비유동 구간 이자부 차입 합(총차입금)을 만든다 — 06의 E 규칙(위치 방식, 유동성장기부채 포함, 이중 계산 제거)을 대상 기간만 넓혀 그대로 적용 |
| 읽는 파일 | `07_계정확보/06_노출도E_만들기.py`(함수를 경로로 불러옴), 저장된 응답, `outputs/04_본수집/` |
| 만드는 파일 | `outputs/07_분석/모듈별/차입_전기간.csv` |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 58초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

11번 노트북의 M4가 비용률(금융비용 TTM ÷ 평균 총차입금)을 만들 때 이 파일의 총차입금을 씁니다. 그래서 11번보다 **먼저** 실행해야 합니다.

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""M4 준비: 전 기간(2018~2024Q3) 모든 보고서에서 유동 구간·비유동 구간 이자부 차입 합(총차입금)을 만든다.
06_노출도E_만들기.py의 규칙(위치 방식, 유동성장기부채 포함, 합계·세부 행 이중 계산 제거)을 그대로 쓴다. 금액·계산 규칙은 06과 같고 대상 기간만 넓다.
저장: outputs/07_분석/모듈별/차입_전기간.csv
"""
import csv
import importlib.util
import json
import sys
from pathlib import Path
import pandas as pd
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)
sys.path.insert(0, str(HERE.parent / "01_파일럿"))
import config  # noqa: E402

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
spec = importlib.util.spec_from_file_location("e06", HERE.parent / "07_계정확보" / "06_노출도E_만들기.py")
e06 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(e06)
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")
OUT = config.BASE / "outputs" / "07_분석" / "모듈별"
OUT.mkdir(parents=True, exist_ok=True)

#### `main()`

In [3]:
def main():
    chosen = {}
    for r in csv.DictReader(open(e06.LOG, encoding="utf-8-sig")):
        if r["status"] == "000":
            k = (r["corp_code"], int(r["year"]), r["report"])
            if k not in chosen or r["fs_div"] == "CFS":
                chosen[k] = r["fs_div"]
    rows = []
    for i, ((c, y, rep), fs) in enumerate(sorted(chosen.items()), 1):
        d = json.loads((e06.FS_DIR / f"{c}_{y}_{config.REPORTS[rep]}_{fs}.json").read_text(encoding="utf-8"))["list"]
        o = e06.one_report(d)
        rows.append({"corp_code": c, "year": y, "report": rep, "fs": fs, "state": o["state"], "num_C": o["num_C"], "num_N": o["num_N"], "assets": o["assets"]})
        if i % 8000 == 0:
            print(f"  {i:,}개 읽음")
    pd.DataFrame(rows).to_csv(OUT / "차입_전기간.csv", index=False, encoding="utf-8-sig")
    print("저장", len(rows), "행")

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [4]:
main()

  8,000개 읽음


  16,000개 읽음


  24,000개 읽음


  32,000개 읽음


저장 34964 행


### 저장된 결과 미리보기 (맨 위 5행)

위 실행이 만들었거나 이미 저장돼 있던 결과 파일을 표로 엽니다. 파일 전체는 엑셀로 여세요.

In [5]:
import pandas as pd
from IPython.display import display

_p = HERE.parents[1] / "outputs/07_분석/모듈별/차입_전기간.csv"
print(_p.name, "|", end=" ")
_df = pd.read_csv(_p, encoding="utf-8-sig", low_memory=False)
print(f"{len(_df):,}행 × {_df.shape[1]}열")
display(_df.head(5))

차입_전기간.csv | 34,964행 × 8열


,corp_code,year,report,fs,state,num_C,num_N,assets
0,100601,2018,FY,CFS,DEBT,2.444344e+10,0.000000e+00,9.384558e+10
1,100601,2018,H1,CFS,DEBT,2.161394e+10,3.017151e+09,1.042823e+11
2,100601,2018,Q1,CFS,DEBT,6.135970e+09,1.011476e+10,9.137234e+10
3,100601,2018,Q3,CFS,DEBT,2.143356e+10,3.120668e+09,1.062916e+11
4,100601,2019,FY,CFS,DEBT,1.386471e+10,3.320000e+09,5.660206e+10
